# 01 — Positive Landslide Inventory Audit

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

## Purpose
This notebook audits the two positive landslide inventories **before any model training**.

It will:

1. Load NASA COOLR, JMG and the Malaysia boundary.
2. Check row counts, schema and missing values.
3. Validate COOLR coordinates and Malaysia-boundary membership.
4. Audit JMG coordinate quality without inventing coordinates.
5. Reconstruct a **date candidate only when source day/month/year fields are complete and valid**.
6. Flag within-source duplicate candidates.
7. Screen cross-source duplicate candidates only where valid WGS84 coordinates exist.
8. Summarise spatial coverage.
9. Save audit outputs into the development workspace.

## Important rules used here
- Raw files are read only.
- Missing JMG values are not filled.
- JMG `RSOx/RSOy` are not converted unless the source CRS is explicitly confirmed.
- No record is automatically deleted as a duplicate.
- JMG dates are not used for rainfall merely because day/month/year fields exist; rainfall eligibility is decided later.
- JMG supporting hazard layers are not treated as landslide occurrence records.


In [1]:
# OPTIONAL: run this only if your environment is missing packages.
# In Google Colab, uncomment and run:
# !pip install -q geopandas pyproj shapely matplotlib pandas numpy

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import Point

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


pandas: 3.0.2
geopandas: 1.1.4


## 1. Project paths

Change **only `PROJECT_ROOT`** to the location of your main `Landslide_FYP` folder.

Expected structure:

```text
Landslide_FYP/
├── Landslide_FYP_Data/
│   ├── 01_Inventory/
│   │   ├── JMG/JMG_Government_Raw_Landslide_Inventory.csv
│   │   └── NASA_COOLR/coolr_malaysia_candidates.geojson
│   └── 02_Boundary/mys_admin0.geojson
└── Landslide_FYP_Development/
    ├── 01_preprocessing/
    ├── 02_cleaned_inventory/
    ├── 03_derived_features/
    ├── 04_model_ready/
    ├── 05_models/
    ├── 06_evaluation/
    └── 07_dashboard_outputs/
```


In [2]:
# ============================================================
# CHANGE THIS ONE LINE
# ============================================================
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

JMG_PATH = RAW_DIR / "01_Inventory" / "JMG" / "JMG_Government_Raw_Landslide_Inventory.csv"
COOLR_PATH = RAW_DIR / "01_Inventory" / "NASA_COOLR" / "coolr_malaysia_candidates.geojson"
BOUNDARY_PATH = RAW_DIR / "02_Boundary" / "mys_admin0.geojson"

PREPROCESS_DIR = DEV_DIR / "01_preprocessing"
CLEAN_DIR = DEV_DIR / "02_cleaned_inventory"
EVAL_DIR = DEV_DIR / "06_evaluation"
AUDIT_DIR = PREPROCESS_DIR / "inventory_audit_outputs"

for folder in [
    PREPROCESS_DIR,
    CLEAN_DIR,
    DEV_DIR / "03_derived_features",
    DEV_DIR / "04_model_ready",
    DEV_DIR / "05_models",
    EVAL_DIR,
    DEV_DIR / "07_dashboard_outputs",
    AUDIT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

for name, path in {
    "JMG": JMG_PATH,
    "COOLR": COOLR_PATH,
    "Malaysia boundary": BOUNDARY_PATH,
}.items():
    print(f"{name}: {path}")
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {name} file:\\n{path}\\n\\n"
            "Check PROJECT_ROOT and your raw-data folder names."
        )

print("\\nAll three required raw files were found.")


JMG: C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\JMG_Government_Raw_Landslide_Inventory.csv
COOLR: C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\NASA_COOLR\coolr_malaysia_candidates.geojson
Malaysia boundary: C:\Landslide_FYP\Landslide_FYP_Data\02_Boundary\mys_admin0.geojson
\nAll three required raw files were found.


## 2. Load the three raw datasets

In [3]:
jmg_raw = pd.read_csv(JMG_PATH)
coolr_raw = gpd.read_file(COOLR_PATH)
malaysia = gpd.read_file(BOUNDARY_PATH)

print("JMG shape:", jmg_raw.shape)
print("COOLR shape:", coolr_raw.shape)
print("Boundary shape:", malaysia.shape)
print()
print("COOLR CRS:", coolr_raw.crs)
print("Boundary CRS:", malaysia.crs)
print("COOLR geometry types:", coolr_raw.geom_type.value_counts().to_dict())
print("Boundary geometry types:", malaysia.geom_type.value_counts().to_dict())


JMG shape: (52, 41)
COOLR shape: (236, 31)
Boundary shape: (1, 18)

COOLR CRS: EPSG:4326
Boundary CRS: EPSG:4326
COOLR geometry types: {'Point': 236}
Boundary geometry types: {'MultiPolygon': 1}


In [4]:
print("JMG columns:")
display(pd.DataFrame({"column": jmg_raw.columns, "dtype": jmg_raw.dtypes.astype(str).values}))

print("\nCOOLR columns:")
display(pd.DataFrame({"column": coolr_raw.columns, "dtype": coolr_raw.dtypes.astype(str).values}))

print("\nBoundary columns:")
display(pd.DataFrame({"column": malaysia.columns, "dtype": malaysia.dtypes.astype(str).values}))


JMG columns:


,column,dtype
0,SOURCE_FILE,str
1,Bil,float64
2,Nama_Pegaw,str
3,Tindakan_d,str
4,Jabatan,str
5,Lain_Lain_,str
6,ID_Tanah_R,str
7,RSOx,float64
8,RSOy,float64
9,WGSe,float64



COOLR columns:


,column,dtype
0,objectid,int32
1,source_name,str
2,source_link,str
3,event_id,int32
4,event_date,int64
5,event_time,str
6,event_title,str
7,event_description,str
8,location_description,str
9,location_accuracy,str



Boundary columns:


,column,dtype
0,iso2,str
1,iso3,str
2,adm0_name,str
3,adm0_name1,object
4,adm0_name2,object
5,adm0_name3,object
6,adm0_pcode,str
7,valid_on,datetime64[ms]
8,valid_to,object
9,version,str


## 3. Preserve source identity

These IDs are created only for the development audit. They do **not** overwrite any original source ID.


In [5]:
jmg = jmg_raw.copy()
coolr = coolr_raw.copy()

jmg.insert(0, "audit_id", [f"JMG_{i:03d}" for i in range(1, len(jmg) + 1)])
coolr.insert(0, "audit_id", [f"COOLR_{i:03d}" for i in range(1, len(coolr) + 1)])

jmg["source_dataset"] = "JMG"
coolr["source_dataset"] = "NASA_COOLR"

print("Development-only audit IDs created.")


Development-only audit IDs created.


## 4. Missing-value audit

In [6]:
def missing_report(df, dataset_name):
    n = len(df)
    report = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "missing_count": [df[c].isna().sum() for c in df.columns],
    })
    report["missing_pct"] = (report["missing_count"] / n * 100).round(2)
    return report.sort_values(["missing_pct", "column"], ascending=[False, True]).reset_index(drop=True)

jmg_missing = missing_report(jmg, "JMG")
coolr_missing = missing_report(coolr.drop(columns="geometry"), "NASA_COOLR")

display(jmg_missing)
display(coolr_missing)

jmg_missing.to_csv(AUDIT_DIR / "jmg_missing_report.csv", index=False)
coolr_missing.to_csv(AUDIT_DIR / "coolr_missing_report.csv", index=False)


,dataset,column,missing_count,missing_pct
0,JMG,Bilangan_M,51,98.08
1,JMG,Lain_Lain_,51,98.08
2,JMG,ID__,50,96.15
3,JMG,Punca__,49,94.23
4,JMG,Sumber,49,94.23
5,JMG,Nama_Pemil,48,92.31
6,JMG,FID,47,90.38
7,JMG,Tindakan_d,31,59.62
8,JMG,Jenis_Kero,22,42.31
9,JMG,Catatan,20,38.46


,dataset,column,missing_count,missing_pct
0,NASA_COOLR,last_edited_date,236,100.00
1,NASA_COOLR,storm_name,235,99.58
2,NASA_COOLR,comments,220,93.22
3,NASA_COOLR,photo_link,171,72.46
4,NASA_COOLR,injury_count,106,44.92
5,NASA_COOLR,event_time,99,41.95
6,NASA_COOLR,event_import_id,66,27.97
7,NASA_COOLR,fatality_count,42,17.80
8,NASA_COOLR,location_description,2,0.85
9,NASA_COOLR,admin_division_name,0,0.00


## 5. JMG coordinate-quality audit

The CSV contains:
- `WGSe`, `WGSn` — apparent longitude/latitude fields.
- `RSOx`, `RSOy` — projected-coordinate fields.

We **do not assume** the CRS of `RSOx/RSOy`.

A WGS84 coordinate is marked directly usable only when:
1. longitude/latitude are within a broad Malaysia envelope, and
2. the pair contains actual decimal precision rather than only whole-degree placeholders.

This intentionally treats values such as `102, 3`, `101, 4`, `100, 5`, and `0, 0` as **not precise enough for point-level modelling**.


In [7]:
required_jmg = ["SOURCE_FILE", "ID_Tanah_R", "RSOx", "RSOy", "WGSe", "WGSn",
                "Hari", "Bulan", "Tahun", "Negeri", "Daerah"]
missing_cols = [c for c in required_jmg if c not in jmg.columns]
if missing_cols:
    raise KeyError(f"Expected JMG columns not found: {missing_cols}")

for c in ["WGSe", "WGSn", "RSOx", "RSOy"]:
    jmg[c] = pd.to_numeric(jmg[c], errors="coerce")

plausible_malaysia = (
    jmg["WGSe"].between(99.0, 120.0, inclusive="both")
    & jmg["WGSn"].between(0.0, 8.5, inclusive="both")
)

has_fractional_precision = (
    ((jmg["WGSe"] % 1).abs() > 1e-6)
    | ((jmg["WGSn"] % 1).abs() > 1e-6)
)

jmg["wgs84_point_usable"] = plausible_malaysia & has_fractional_precision

def coordinate_status(row):
    if pd.isna(row["WGSe"]) or pd.isna(row["WGSn"]):
        return "MISSING_WGS84"
    if row["WGSe"] == 0 and row["WGSn"] == 0:
        return "ZERO_PLACEHOLDER"
    if not (99 <= row["WGSe"] <= 120 and 0 <= row["WGSn"] <= 8.5):
        return "OUTSIDE_MALAYSIA_ENVELOPE"
    if not row["wgs84_point_usable"]:
        return "WHOLE_DEGREE_NOT_PRECISE"
    return "USABLE_DECIMAL_WGS84"

jmg["coordinate_status"] = jmg.apply(coordinate_status, axis=1)

print("JMG coordinate status:")
display(jmg["coordinate_status"].value_counts(dropna=False).rename_axis("status").reset_index(name="count"))

print("\nBy source file:")
display(pd.crosstab(jmg["SOURCE_FILE"], jmg["coordinate_status"], margins=True))

print("\nIMPORTANT:")
print("RSOx/RSOy have been preserved but NOT transformed.")
print("Their source CRS must be confirmed from the original .prj/shapefile metadata first.")


JMG coordinate status:


,status,count
0,WHOLE_DEGREE_NOT_PRECISE,39
1,ZERO_PLACEHOLDER,9
2,USABLE_DECIMAL_WGS84,4



By source file:


coordinate_status,USABLE_DECIMAL_WGS84,WHOLE_DEGREE_NOT_PRECISE,ZERO_PLACEHOLDER,All
SOURCE_FILE,,,,
ch_TANAH_RUNTUH_SEMENANJUNG_.shp,0,11,5,16
ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp,0,0,1,1
kdsg_TANAH_RUNTUH_SABAH.shp,4,0,0,4
pp_TANAH_RUNTUH_SEMENANJUNG_.shp,0,3,0,3
uk_TANAH_RUNTUH_SEMENANJUNG_.shp,0,25,3,28
All,4,39,9,52



IMPORTANT:
RSOx/RSOy have been preserved but NOT transformed.
Their source CRS must be confirmed from the original .prj/shapefile metadata first.


## 6. JMG source-date completeness

This does **not** declare rainfall eligibility.

A `source_date_candidate` is created only when JMG supplied a valid day, recognised Malay/English month and valid year.  
Rows with `Hari = 0`, missing day, unrecognised month, or invalid date remain `NaT`.

Later, event-specific rainfall should only use records whose event date is judged reliable enough for that purpose.


In [8]:
month_map = {
    "januari": 1, "january": 1,
    "februari": 2, "february": 2,
    "mac": 3, "march": 3,
    "april": 4,
    "mei": 5, "may": 5,
    "jun": 6, "june": 6,
    "julai": 7, "july": 7,
    "ogos": 8, "august": 8,
    "september": 9,
    "oktober": 10, "october": 10,
    "november": 11,
    "disember": 12, "december": 12,
}

jmg["_month_num"] = (
    jmg["Bulan"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(month_map)
)

jmg["_day_num"] = pd.to_numeric(jmg["Hari"], errors="coerce")
jmg["_year_num"] = pd.to_numeric(jmg["Tahun"], errors="coerce")

date_parts_complete = (
    jmg["_day_num"].between(1, 31)
    & jmg["_month_num"].between(1, 12)
    & jmg["_year_num"].between(1900, 2100)
)

jmg["source_date_candidate"] = pd.NaT

candidate_dates = pd.to_datetime(
    pd.DataFrame({
        "year": jmg.loc[date_parts_complete, "_year_num"],
        "month": jmg.loc[date_parts_complete, "_month_num"],
        "day": jmg.loc[date_parts_complete, "_day_num"],
    }),
    errors="coerce"
)

jmg.loc[date_parts_complete, "source_date_candidate"] = candidate_dates.values
jmg["source_date_complete_valid"] = jmg["source_date_candidate"].notna()

print("Valid complete source date fields:", int(jmg["source_date_complete_valid"].sum()))
print("Not complete/valid:", int((~jmg["source_date_complete_valid"]).sum()))

display(
    jmg[
        ["audit_id", "SOURCE_FILE", "ID_Tanah_R", "Hari", "Bulan", "Tahun",
         "source_date_candidate", "source_date_complete_valid"]
    ].head(20)
)


Valid complete source date fields: 47
Not complete/valid: 5


,audit_id,SOURCE_FILE,ID_Tanah_R,Hari,Bulan,Tahun,source_date_candidate,source_date_complete_valid
0,JMG_001,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1318_M_05_SF_01,18.0,Disember,2021.0,2021-12-18,True
1,JMG_002,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1318_N_01_L_01,18.0,Disember,2021.0,2021-12-18,True
2,JMG_003,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1322_M_02_SF_01,19.0,Disember,2021.0,2021-12-19,True
3,JMG_004,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1324_M_02_SF_01,18.0,Disember,2021.0,2021-12-18,True
4,JMG_005,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1324_M_03_SF_01,18.0,Disember,2021.0,2021-12-18,True
5,JMG_006,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1324_M_04_SF_01,18.0,Disember,2021.0,2021-12-18,True
6,JMG_007,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1324_N_01_L_01,18.0,Disember,2021.0,2021-12-18,True
7,JMG_008,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1324_N_05_L_01,18.0,Disember,2021.0,2021-12-18,True
8,JMG_009,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1324_N_06_L_01,18.0,Disember,2021.0,2021-12-18,True
9,JMG_010,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,AU1415_N_01_L_01,18.0,Disember,2021.0,2021-12-18,True


## 7. COOLR core-field and date audit

COOLR event dates are stored as Unix epoch milliseconds in the current GeoJSON.


In [9]:
required_coolr = [
    "event_id", "event_date", "latitude", "longitude", "location_accuracy",
    "country_code", "admin_division_name", "geometry"
]
missing_cols = [c for c in required_coolr if c not in coolr.columns]
if missing_cols:
    raise KeyError(f"Expected COOLR columns not found: {missing_cols}")

coolr["event_date_parsed"] = pd.to_datetime(
    coolr["event_date"], unit="ms", errors="coerce", utc=True
)

coolr["coordinate_fields_valid"] = (
    pd.to_numeric(coolr["longitude"], errors="coerce").between(99, 120)
    & pd.to_numeric(coolr["latitude"], errors="coerce").between(0, 8.5)
)

core_summary = pd.DataFrame({
    "field": ["event_id", "event_date", "latitude", "longitude",
              "location_accuracy", "country_code", "admin_division_name"],
    "missing_count": [
        coolr[c].isna().sum()
        for c in ["event_id", "event_date", "latitude", "longitude",
                  "location_accuracy", "country_code", "admin_division_name"]
    ]
})

display(core_summary)
print("Parsed COOLR dates:", coolr["event_date_parsed"].notna().sum(), "/", len(coolr))
print("Date range:", coolr["event_date_parsed"].min(), "to", coolr["event_date_parsed"].max())
print("Coordinate-field envelope valid:", coolr["coordinate_fields_valid"].sum(), "/", len(coolr))


,field,missing_count
0,event_id,0
1,event_date,0
2,latitude,0
3,longitude,0
4,location_accuracy,0
5,country_code,0
6,admin_division_name,0


Parsed COOLR dates: 236 / 236
Date range: 2007-03-22 00:00:00+00:00 to 2022-03-09 00:00:00+00:00
Coordinate-field envelope valid: 236 / 236


## 8. Malaysia-boundary validation for COOLR

In [10]:
if coolr.crs is None:
    raise ValueError("COOLR CRS is missing. Do not assume EPSG:4326 without checking the source.")
if malaysia.crs is None:
    raise ValueError("Malaysia boundary CRS is missing.")

coolr_for_boundary = coolr.to_crs(malaysia.crs)

# Boundary file currently contains one Malaysia MultiPolygon.
# If a future boundary file contains multiple features, dissolve it first.
if len(malaysia) == 1:
    malaysia_geom = malaysia.geometry.iloc[0]
else:
    malaysia_geom = malaysia.dissolve().geometry.iloc[0]

coolr["inside_malaysia_boundary"] = coolr_for_boundary.geometry.apply(malaysia_geom.covers).values

print("COOLR boundary result:")
display(
    coolr["inside_malaysia_boundary"]
    .value_counts(dropna=False)
    .rename_axis("inside_boundary")
    .reset_index(name="count")
)

outside = coolr.loc[
    ~coolr["inside_malaysia_boundary"],
    ["audit_id", "event_id", "event_title", "latitude", "longitude",
     "admin_division_name"]
]
if len(outside):
    print("Review these points:")
    display(outside)
else:
    print("All COOLR points are covered by the supplied Malaysia boundary.")


COOLR boundary result:


,inside_boundary,count
0,True,236


All COOLR points are covered by the supplied Malaysia boundary.


## 9. Within-source duplicate screening

These are **candidate duplicate flags**, not deletion instructions.

### JMG checks
- repeated non-null `ID_Tanah_R`
- repeated exact `RSOx/RSOy`

### COOLR checks
- repeated `event_id`
- repeated exact latitude/longitude


In [11]:
# ---------- JMG ----------
jmg["duplicate_jmg_id"] = (
    jmg["ID_Tanah_R"].notna()
    & jmg["ID_Tanah_R"].duplicated(keep=False)
)

jmg["duplicate_jmg_exact_rso"] = (
    jmg[["RSOx", "RSOy"]].notna().all(axis=1)
    & jmg.duplicated(["RSOx", "RSOy"], keep=False)
)

jmg_dup_candidates = jmg.loc[
    jmg["duplicate_jmg_id"] | jmg["duplicate_jmg_exact_rso"],
    [
        "audit_id", "SOURCE_FILE", "ID_Tanah_R", "RSOx", "RSOy",
        "WGSe", "WGSn", "Negeri", "Daerah",
        "source_date_candidate",
        "duplicate_jmg_id", "duplicate_jmg_exact_rso"
    ]
].sort_values(["SOURCE_FILE", "ID_Tanah_R"], na_position="last")

print("JMG candidate duplicate rows:", len(jmg_dup_candidates))
display(jmg_dup_candidates)

# ---------- COOLR ----------
coolr["duplicate_coolr_event_id"] = coolr["event_id"].duplicated(keep=False)

coolr["duplicate_coolr_exact_coord"] = (
    coolr[["latitude", "longitude"]].notna().all(axis=1)
    & coolr.duplicated(["latitude", "longitude"], keep=False)
)

coolr_dup_candidates = coolr.loc[
    coolr["duplicate_coolr_event_id"] | coolr["duplicate_coolr_exact_coord"],
    [
        "audit_id", "event_id", "event_date_parsed", "event_title",
        "location_description", "location_accuracy",
        "latitude", "longitude",
        "duplicate_coolr_event_id", "duplicate_coolr_exact_coord"
    ]
].sort_values(["latitude", "longitude"])

print("\nCOOLR candidate duplicate rows:", len(coolr_dup_candidates))
display(coolr_dup_candidates)


JMG candidate duplicate rows: 6


,audit_id,SOURCE_FILE,ID_Tanah_R,RSOx,RSOy,WGSe,WGSn,Negeri,Daerah,source_date_candidate,duplicate_jmg_id,duplicate_jmg_exact_rso
36,JMG_037,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,AY44201_LS_05,376296.00,497079.0,0.0,0.0,Pahang,Cameron Highlands,2021-12-19,True,False
37,JMG_038,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,AY44201_LS_05,376020.00,495378.0,101.0,4.0,Pahang,Cameron Highlands,2021-12-19,True,False
41,JMG_042,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,AY44201_M_01_SF_01,377057.00,497079.0,0.0,0.0,Pahang,Cameron Highlands,2021-05-29,False,True
42,JMG_043,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,AY44201_M_02_SF_01,377057.00,497079.0,0.0,0.0,Pahang,Cameron Highlands,2021-01-06,False,True
48,JMG_049,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,FH2101_M_01_SF_01,256719.82,603963.5,100.0,5.0,Pulau Pinang,Timor Laut,2021-10-30,True,False
49,JMG_050,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,FH2101_M_01_SF_01,256544.00,604104.0,100.0,5.0,Pulau Pinang,Timor Laut,2021-09-28,True,False



COOLR candidate duplicate rows: 2


,audit_id,event_id,event_date_parsed,event_title,location_description,location_accuracy,latitude,longitude,duplicate_coolr_event_id,duplicate_coolr_exact_coord
0,COOLR_001,9083,2015-12-19 00:00:00+00:00,Labuan,Labuan,25km,5.3149,115.2203,False,True
11,COOLR_012,9082,2015-12-19 00:00:00+00:00,Labuan,Labuan,25km,5.3149,115.2203,False,True


## 10. Cross-source duplicate screening — only valid WGS84 JMG points

This screening uses only JMG rows currently marked `USABLE_DECIMAL_WGS84`.

**No JMG point with whole-degree/zero placeholder coordinates is used here.**

Distance alone does not prove a duplicate. The output is only a candidate-review table.  
Later, once the CRS of each JMG `RSOx/RSOy` source is confirmed and all records can be safely converted to WGS84, rerun this section with the complete JMG point inventory.


In [12]:
EARTH_RADIUS_KM = 6371.0088

def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

jmg_valid = jmg.loc[jmg["wgs84_point_usable"]].copy()

pairs = []

for _, jr in jmg_valid.iterrows():
    distances = haversine_km(
        jr["WGSn"],
        jr["WGSe"],
        coolr["latitude"].to_numpy(dtype=float),
        coolr["longitude"].to_numpy(dtype=float),
    )

    # Keep nearest 5 COOLR records for audit.
    nearest_idx = np.argsort(distances)[:5]

    for idx in nearest_idx:
        cr = coolr.iloc[idx]
        d = float(distances[idx])

        if d <= 10.0:
            pairs.append({
                "jmg_audit_id": jr["audit_id"],
                "jmg_source_file": jr["SOURCE_FILE"],
                "jmg_id": jr["ID_Tanah_R"],
                "jmg_date_candidate": jr["source_date_candidate"],
                "jmg_state": jr["Negeri"],
                "jmg_district": jr["Daerah"],
                "jmg_lat": jr["WGSn"],
                "jmg_lon": jr["WGSe"],
                "coolr_audit_id": cr["audit_id"],
                "coolr_event_id": cr["event_id"],
                "coolr_date": cr["event_date_parsed"],
                "coolr_title": cr.get("event_title"),
                "coolr_location": cr.get("location_description"),
                "coolr_location_accuracy": cr.get("location_accuracy"),
                "coolr_lat": cr["latitude"],
                "coolr_lon": cr["longitude"],
                "distance_km": round(d, 3),
                "within_1km": d <= 1,
                "within_5km": d <= 5,
                "within_10km": d <= 10,
            })

cross_source_candidates = pd.DataFrame(pairs)

print("JMG rows eligible for current WGS84 cross-source screening:", len(jmg_valid))
print("Candidate pairs within 10 km:", len(cross_source_candidates))

if len(cross_source_candidates):
    cross_source_candidates = cross_source_candidates.sort_values("distance_km").reset_index(drop=True)
    display(cross_source_candidates)
else:
    print("No candidate pairs within 10 km among currently usable JMG WGS84 points.")


JMG rows eligible for current WGS84 cross-source screening: 4
Candidate pairs within 10 km: 20


,jmg_audit_id,jmg_source_file,jmg_id,jmg_date_candidate,jmg_state,jmg_district,jmg_lat,jmg_lon,coolr_audit_id,coolr_event_id,coolr_date,coolr_title,coolr_location,coolr_location_accuracy,coolr_lat,coolr_lon,distance_km,within_1km,within_5km,within_10km
0,JMG_048,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,2021-01-20,Sabah,Ranau,6.005770,116.539150,COOLR_184,3349,2011-04-10 09:00:00+00:00,"Kg Mohimboyon(?), Kundasang(?), 34 km along ro...","Kg Mohimboyon(?), Kundasang(?), 34 km along ro...",10km,6.0090,116.5268,1.412,False,True,True
1,JMG_046,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,6.012806,116.600528,COOLR_099,290,2007-10-03 00:00:00+00:00,"Kundasang-Tamparuli road, Kundasang, near the ...","Kundasang-Tamparuli road, Kundasang, near the ...",50km,5.9975,116.5956,1.787,False,True,True
2,JMG_047,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,6.014236,116.601917,COOLR_099,290,2007-10-03 00:00:00+00:00,"Kundasang-Tamparuli road, Kundasang, near the ...","Kundasang-Tamparuli road, Kundasang, near the ...",50km,5.9975,116.5956,1.988,False,True,True
3,JMG_045,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,5.999077,116.541813,COOLR_184,3349,2011-04-10 09:00:00+00:00,"Kg Mohimboyon(?), Kundasang(?), 34 km along ro...","Kg Mohimboyon(?), Kundasang(?), 34 km along ro...",10km,6.0090,116.5268,1.993,False,True,True
4,JMG_047,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,6.014236,116.601917,COOLR_103,7639,2015-04-10 00:00:00+00:00,Mesilau,Mesilau,5km,6.0462,116.5964,3.606,False,True,True
5,JMG_046,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,6.012806,116.600528,COOLR_004,5120,2013-07-16 00:00:00+00:00,"Kampung Mesilau, Kundasang, Borneo","Kampung Mesilau, Kundasang, Borneo",5km,5.9913,116.5754,3.666,False,True,True
6,JMG_046,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,6.012806,116.600528,COOLR_103,7639,2015-04-10 00:00:00+00:00,Mesilau,Mesilau,5km,6.0462,116.5964,3.741,False,True,True
7,JMG_045,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,5.999077,116.541813,COOLR_004,5120,2013-07-16 00:00:00+00:00,"Kampung Mesilau, Kundasang, Borneo","Kampung Mesilau, Kundasang, Borneo",5km,5.9913,116.5754,3.814,False,True,True
8,JMG_047,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,6.014236,116.601917,COOLR_004,5120,2013-07-16 00:00:00+00:00,"Kampung Mesilau, Kundasang, Borneo","Kampung Mesilau, Kundasang, Borneo",5km,5.9913,116.5754,3.886,False,True,True
9,JMG_045,kdsg_TANAH_RUNTUH_SABAH.shp,NaN,NaT,Sabah,Ranau,5.999077,116.541813,COOLR_233,3348,2011-04-10 03:00:00+00:00,"Kg Dumpiring Atas, 15 km from Ranau","Kg Dumpiring Atas, 15 km from Ranau",5km,5.9840,116.5742,3.955,False,True,True


## 11. Spatial coverage summary

At this stage:
- COOLR can be mapped nationally.
- Only JMG records with valid decimal WGS84 coordinates are plotted.
- Unresolved JMG records remain in the audit table and are **not discarded**.


In [13]:
coolr_state_counts = (
    coolr["admin_division_name"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("admin_division_name")
    .reset_index(name="coolr_count")
)

jmg_state_counts = (
    jmg["Negeri"]
    .fillna("MISSING")
    .value_counts()
    .rename_axis("Negeri")
    .reset_index(name="jmg_raw_count")
)

print("COOLR by admin division:")
display(coolr_state_counts)

print("\nJMG raw records by supplied state field:")
display(jmg_state_counts)

coolr_state_counts.to_csv(AUDIT_DIR / "coolr_state_counts.csv", index=False)
jmg_state_counts.to_csv(AUDIT_DIR / "jmg_state_counts.csv", index=False)


COOLR by admin division:


,admin_division_name,coolr_count
0,Sabah,47
1,Sarawak,35
2,Pulau Pinang,34
3,Selangor,33
4,Wilayah Persekutuan Kuala Lumpur,30
5,Pahang,23
6,Perak,11
7,Kelantan,6
8,Wilayah Persekutuan Labuan,4
9,Negeri Sembilan,4



JMG raw records by supplied state field:


,Negeri,jmg_raw_count
0,Selangor,25
1,Pahang,16
2,Sabah,4
3,W.P. Kuala Lumpur,3
4,Pulau Pinang,3
5,Perak,1


In [14]:
# ============================================================
# 11. VISUALISE CURRENT POSITIVE LANDSLIDE INVENTORY
# ============================================================

from IPython.display import display, Image

# Build GeoDataFrame only from JMG rows whose WGS84
# coordinates are currently usable.
jmg_plot = jmg.loc[jmg["wgs84_point_usable"]].copy()

jmg_plot = gpd.GeoDataFrame(
    jmg_plot,
    geometry=gpd.points_from_xy(
        jmg_plot["WGSe"],
        jmg_plot["WGSn"]
    ),
    crs="EPSG:4326"
)

# Convert everything to the same CRS
malaysia_4326 = malaysia.to_crs("EPSG:4326")
coolr_4326 = coolr.to_crs("EPSG:4326")

# ------------------------------------------------------------
# Create map
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 9))

# Malaysia boundary
malaysia_4326.boundary.plot(
    ax=ax,
    linewidth=0.8
)

# NASA COOLR landslides
coolr_4326.plot(
    ax=ax,
    markersize=18,
    alpha=0.65,
    label="NASA COOLR"
)

# JMG landslides that currently have usable WGS84 coordinates
if len(jmg_plot) > 0:
    jmg_plot.plot(
        ax=ax,
        markersize=45,
        marker="x",
        label="JMG usable WGS84"
    )

# Map formatting
ax.set_title(
    "Positive Landslide Inventory - Current Spatial Audit"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.legend()

ax.grid(
    True,
    alpha=0.2
)

plt.tight_layout()

# ------------------------------------------------------------
# Save map
# ------------------------------------------------------------
map_path = (
    AUDIT_DIR
    / "positive_inventory_spatial_audit.png"
)

fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

# Close matplotlib figure so VS Code does not try
# to use the broken PNG notebook renderer
plt.close(fig)

print("Map successfully created.")
print("Saved to:")
print(map_path)

# ------------------------------------------------------------
# Display saved image using IPython
# ------------------------------------------------------------
print("Open this file manually:")
print(map_path)

Map successfully created.
Saved to:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\inventory_audit_outputs\positive_inventory_spatial_audit.png
Open this file manually:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\inventory_audit_outputs\positive_inventory_spatial_audit.png


## 12. Save audit datasets

In [15]:
# Remove helper-only month/day/year conversion columns before export if desired.
helper_cols = ["_month_num", "_day_num", "_year_num"]

# JMG stays CSV because most JMG rows do not yet have safe WGS84 geometries.
jmg_export = jmg.drop(columns=helper_cols, errors="ignore").copy()
jmg_export.to_csv(AUDIT_DIR / "jmg_inventory_audit.csv", index=False)

# COOLR retains geometry.
coolr.to_file(AUDIT_DIR / "coolr_inventory_audit.geojson", driver="GeoJSON")
coolr.drop(columns="geometry").to_csv(AUDIT_DIR / "coolr_inventory_audit.csv", index=False)

jmg_dup_candidates.to_csv(AUDIT_DIR / "jmg_within_source_duplicate_candidates.csv", index=False)
coolr_dup_candidates.to_csv(AUDIT_DIR / "coolr_within_source_duplicate_candidates.csv", index=False)

if len(cross_source_candidates):
    cross_source_candidates.to_csv(
        AUDIT_DIR / "cross_source_duplicate_candidates_current_wgs84_only.csv",
        index=False
    )
else:
    pd.DataFrame().to_csv(
        AUDIT_DIR / "cross_source_duplicate_candidates_current_wgs84_only.csv",
        index=False
    )

print("Audit outputs written to:")
print(AUDIT_DIR)


Audit outputs written to:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\inventory_audit_outputs


## 13. Generate final Stage-1 audit summary

In [16]:
summary_rows = [
    ("JMG raw rows", len(jmg)),
    ("COOLR raw rows", len(coolr)),
    ("Raw total candidates (NOT unique count)", len(jmg) + len(coolr)),
    ("COOLR inside Malaysia boundary", int(coolr["inside_malaysia_boundary"].sum())),
    ("COOLR outside Malaysia boundary", int((~coolr["inside_malaysia_boundary"]).sum())),
    ("COOLR parsed event dates", int(coolr["event_date_parsed"].notna().sum())),
    ("COOLR repeated event-ID rows", int(coolr["duplicate_coolr_event_id"].sum())),
    ("COOLR repeated exact-coordinate rows", int(coolr["duplicate_coolr_exact_coord"].sum())),
    ("JMG usable decimal WGS84 rows", int(jmg["wgs84_point_usable"].sum())),
    ("JMG WGS84 unresolved/not precise rows", int((~jmg["wgs84_point_usable"]).sum())),
    ("JMG complete valid source-date fields", int(jmg["source_date_complete_valid"].sum())),
    ("JMG repeated non-null ID rows", int(jmg["duplicate_jmg_id"].sum())),
    ("JMG repeated exact RSO-coordinate rows", int(jmg["duplicate_jmg_exact_rso"].sum())),
]

audit_summary = pd.DataFrame(summary_rows, columns=["check", "value"])
display(audit_summary)

audit_summary.to_csv(AUDIT_DIR / "inventory_audit_summary.csv", index=False)

print("\nMODEL-READINESS GATE")
print("--------------------")
print("1. COOLR can proceed to national predictor extraction after duplicate review.")
print("2. JMG cannot yet be fully merged spatially because many WGS84 fields are not point-precise.")
print("3. Do NOT discard those JMG rows.")
print("4. Confirm the CRS of each original JMG source shapefile / .prj and convert RSOx/RSOy safely.")
print("5. After JMG coordinate conversion, rerun cross-source duplicate screening.")
print("6. Do not generate pseudo-absence points until the final positive inventory is spatially resolved.")
print("7. Do not use JMG dates for event rainfall until date reliability is separately reviewed.")


,check,value
0,JMG raw rows,52
1,COOLR raw rows,236
2,Raw total candidates (NOT unique count),288
3,COOLR inside Malaysia boundary,236
4,COOLR outside Malaysia boundary,0
5,COOLR parsed event dates,236
6,COOLR repeated event-ID rows,0
7,COOLR repeated exact-coordinate rows,2
8,JMG usable decimal WGS84 rows,4
9,JMG WGS84 unresolved/not precise rows,48



MODEL-READINESS GATE
--------------------
1. COOLR can proceed to national predictor extraction after duplicate review.
2. JMG cannot yet be fully merged spatially because many WGS84 fields are not point-precise.
3. Do NOT discard those JMG rows.
4. Confirm the CRS of each original JMG source shapefile / .prj and convert RSOx/RSOy safely.
5. After JMG coordinate conversion, rerun cross-source duplicate screening.
6. Do not generate pseudo-absence points until the final positive inventory is spatially resolved.
7. Do not use JMG dates for event rainfall until date reliability is separately reviewed.


## 14. What comes after this notebook

Do **not** train Logistic Regression or Random Forest yet.

The next stage should be:

### `02_jmg_coordinate_resolution.ipynb`
1. Read the original JMG shapefile CRS / `.prj` information.
2. Assign the correct CRS **per source file**.
3. Convert `RSOx/RSOy` to WGS84.
4. Compare converted positions with any available JMG WGS fields.
5. Check Malaysia-boundary membership.
6. Re-run cross-source NASA COOLR ↔ JMG duplicate screening.
7. Produce the final reviewed positive-event inventory.

Only after the positive inventory is resolved should we create pseudo-absence samples and extract environmental predictors.


## 15. Historical land-cover matching readiness (NEW)

This section **does not download land cover** or assert that a dated record actually reflects an occurrence date. It flags which records need date and/or coordinate review before matching land cover and other time-dependent factors.

Coverage windows here are *planning windows*, not proof of cloud-free valid pixels or API availability at each event coordinate:
- Figshare Indo-Malaysian LCLU: 1982–2015 (coarse resolution)
- Copernicus LC100: 2015–2019
- Dynamic World: 2015–2025 (2015 begins mid-year; verify imagery and pixel validity)
- Esri annual land cover: 2017–2025 (check the available year per tile)

**JMG dates are candidates only until independently verified.** Unknown occurrence years cannot be assigned historic predictors safely. Static terrain can still be prepared for coordinates deemed reliable.

In [17]:

# ============================================================
# 15. HISTORICAL LAND COVER & LANDSLIDE RECORD READINESS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 15.1 REUSE ORIGINAL NOTEBOOK VARIABLES AND DIRECTORIES
# ------------------------------------------------------------

# These variables are created in your original audit notebook:
# PROJECT_ROOT, DEV_DIR, AUDIT_DIR, jmg, coolr

required_variables = ["AUDIT_DIR", "jmg", "coolr"]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run original notebook Sections 1-14 first. "
        f"Missing variables: {missing_variables}"
    )

AUDIT_DIR = Path(AUDIT_DIR)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", AUDIT_DIR)
print("JMG records:", len(jmg))
print("NASA COOLR records:", len(coolr))


# ------------------------------------------------------------
# 15.2 PREPARE JMG LANDSLIDE RECORDS
# ------------------------------------------------------------

# JMG date is reconstructed from Hari, Bulan, Tahun.
# It is NOT independently verified as an occurrence date.

jmg_check = jmg.copy()

jmg_dates = pd.to_datetime(
    jmg_check["source_date_candidate"],
    errors="coerce"
)

jmg_audit = pd.DataFrame({
    "audit_id": jmg_check["audit_id"],
    "source_dataset": "JMG",
    "source_record_id": jmg_check["ID_Tanah_R"],
    "source_file": jmg_check["SOURCE_FILE"],

    "state": jmg_check["Negeri"],
    "district": jmg_check["Daerah"],

    "latitude": pd.to_numeric(
        jmg_check["WGSn"], errors="coerce"
    ),
    "longitude": pd.to_numeric(
        jmg_check["WGSe"], errors="coerce"
    ),

    "coordinate_status": jmg_check["coordinate_status"],

    "coordinate_usable": (
        jmg_check["wgs84_point_usable"]
        .fillna(False)
        .astype(bool)
    ),

    "date_candidate": jmg_dates,
    "date_source": "JMG_Hari_Bulan_Tahun",
    "date_verified": False
})

# Preserve useful source information
jmg_audit["original_year"] = jmg_check["Tahun"]
jmg_audit["original_month"] = jmg_check["Bulan"]
jmg_audit["original_day"] = jmg_check["Hari"]


# ------------------------------------------------------------
# 15.3 PREPARE NASA COOLR LANDSLIDE RECORDS
# ------------------------------------------------------------

coolr_check = coolr.copy()

# Original notebook already converted epoch milliseconds
# to datetime in the event_date_parsed column.

coolr_dates = pd.to_datetime(
    coolr_check["event_date_parsed"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

coolr_coordinate_usable = (
    coolr_check["coordinate_fields_valid"]
    .fillna(False)
    .astype(bool)
    &
    coolr_check["inside_malaysia_boundary"]
    .fillna(False)
    .astype(bool)
)

coolr_audit = pd.DataFrame({
    "audit_id": coolr_check["audit_id"],
    "source_dataset": "NASA_COOLR",
    "source_record_id": coolr_check["event_id"],
    "source_file": pd.NA,

    "state": coolr_check["admin_division_name"],
    "district": pd.NA,

    "latitude": pd.to_numeric(
        coolr_check["latitude"], errors="coerce"
    ),
    "longitude": pd.to_numeric(
        coolr_check["longitude"], errors="coerce"
    ),

    "coordinate_status": np.where(
        coolr_coordinate_usable,
        "VALID_INSIDE_MALAYSIA",
        "COORDINATE_REVIEW_REQUIRED"
    ),

    "coordinate_usable": coolr_coordinate_usable,

    "date_candidate": coolr_dates,
    "date_source": "NASA_COOLR_event_date",
    "date_verified": False
})

coolr_audit["original_year"] = coolr_dates.dt.year
coolr_audit["original_month"] = coolr_dates.dt.month
coolr_audit["original_day"] = coolr_dates.dt.day


# ------------------------------------------------------------
# 15.4 COMBINE JMG + NASA COOLR
# ------------------------------------------------------------

historical_audit = pd.concat(
    [jmg_audit, coolr_audit],
    ignore_index=True
)

historical_audit["event_year_candidate"] = (
    historical_audit["date_candidate"]
    .dt.year
    .astype("Int64")
)

historical_audit["date_available"] = (
    historical_audit["date_candidate"].notna()
)

print("\nCombined record count:", len(historical_audit))


# ------------------------------------------------------------
# 15.5 CHECK HISTORICAL LAND COVER YEAR AVAILABILITY
# ------------------------------------------------------------

year = historical_audit["event_year_candidate"]

# Year coverage only - not actual raster matching.
historical_audit["figshare_1982_2015"] = (
    year.between(1982, 2015).fillna(False)
)

historical_audit["copernicus_2015_2019"] = (
    year.between(2015, 2019).fillna(False)
)

historical_audit["dynamic_world_2015_2025"] = (
    year.between(2015, 2025).fillna(False)
)

historical_audit["esri_2017_2025"] = (
    year.between(2017, 2025).fillna(False)
)

# Dynamic World starts during 2015, not January 1.
historical_audit["dynamic_world_partial_2015"] = (
    year.eq(2015).fillna(False)
)

landcover_columns = [
    "figshare_1982_2015",
    "copernicus_2015_2019",
    "dynamic_world_2015_2025",
    "esri_2017_2025"
]

historical_audit["has_landcover_year_match"] = (
    historical_audit[landcover_columns].any(axis=1)
)


# ------------------------------------------------------------
# 15.6 DETERMINE REVIEW REQUIREMENTS
# ------------------------------------------------------------

historical_audit["temporal_review_needed"] = (
    ~historical_audit["date_available"]
    | ~historical_audit["date_verified"]
)

historical_audit["spatial_review_needed"] = (
    ~historical_audit["coordinate_usable"]
)

historical_audit["needs_review"] = (
    historical_audit["temporal_review_needed"]
    | historical_audit["spatial_review_needed"]
)

def get_review_reason(row):

    reasons = []

    if not row["date_available"]:
        reasons.append("MISSING_OR_INVALID_DATE")

    elif not row["date_verified"]:
        reasons.append("DATE_NOT_VERIFIED")

    if not row["coordinate_usable"]:
        reasons.append("COORDINATE_NOT_READY")

    if (
        row["date_available"]
        and not row["has_landcover_year_match"]
    ):
        reasons.append("NO_LANDCOVER_YEAR_MATCH")

    if row["dynamic_world_partial_2015"]:
        reasons.append("DYNAMIC_WORLD_2015_PARTIAL")

    if not reasons:
        return "NO_FLAG"

    return "; ".join(reasons)


historical_audit["review_reason"] = (
    historical_audit.apply(get_review_reason, axis=1)
)

# Candidate matching categories
# These do not indicate that dates are verified.

historical_audit["matching_status"] = np.select(
    [
        ~historical_audit["coordinate_usable"],
        ~historical_audit["date_available"],
        ~historical_audit["has_landcover_year_match"]
    ],
    [
        "COORDINATE_REVIEW",
        "DATE_REVIEW",
        "NO_HISTORICAL_YEAR_MATCH"
    ],
    default="CANDIDATE_MATCH_REQUIRES_DATE_VERIFICATION"
)


# ------------------------------------------------------------
# 15.7 EXPORT ALL RECORDS
# ------------------------------------------------------------

all_records_path = (
    AUDIT_DIR /
    "landslide_historical_landcover_readiness_all_records.csv"
)

historical_audit.to_csv(
    all_records_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 15.8 EXPORT RECORDS REQUIRING REVIEW
# ------------------------------------------------------------

review_path = (
    AUDIT_DIR /
    "landslide_records_needing_date_or_coordinate_review.csv"
)

historical_audit.loc[
    historical_audit["needs_review"]
].to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 15.9 EXPORT EVENT YEAR DISTRIBUTION
# ------------------------------------------------------------

year_distribution = (
    historical_audit
    .groupby(
        ["source_dataset", "event_year_candidate"],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["source_dataset", "event_year_candidate"],
        na_position="last"
    )
)

year_path = (
    AUDIT_DIR /
    "landslide_candidate_year_distribution.csv"
)

year_distribution.to_csv(
    year_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 15.10 EXPORT LAND COVER YEAR COVERAGE SUMMARY
# ------------------------------------------------------------

coverage_summary = (
    historical_audit
    .groupby("source_dataset")
    .agg(
        total_records=("audit_id", "size"),
        dates_available=("date_available", "sum"),
        usable_coordinates=("coordinate_usable", "sum"),
        records_needing_review=("needs_review", "sum"),
        figshare_1982_2015=("figshare_1982_2015", "sum"),
        copernicus_2015_2019=("copernicus_2015_2019", "sum"),
        dynamic_world_2015_2025=(
            "dynamic_world_2015_2025", "sum"
        ),
        esri_2017_2025=("esri_2017_2025", "sum")
    )
    .reset_index()
)

coverage_path = (
    AUDIT_DIR /
    "landslide_candidate_landcover_year_coverage_summary.csv"
)

coverage_summary.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 15.11 DISPLAY FINAL AUDIT RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("HISTORICAL LAND COVER READINESS AUDIT")
print("=" * 60)

print("Total records:", len(historical_audit))
print(
    "JMG records:",
    (historical_audit["source_dataset"] == "JMG").sum()
)
print(
    "NASA COOLR records:",
    (historical_audit["source_dataset"] == "NASA_COOLR").sum()
)
print(
    "Records with candidate dates:",
    int(historical_audit["date_available"].sum())
)
print(
    "Records with tentatively usable coordinates:",
    int(historical_audit["coordinate_usable"].sum())
)
print(
    "Records with land cover year overlap:",
    int(historical_audit["has_landcover_year_match"].sum())
)

print("\nYear distribution:")
print(year_distribution.to_string(index=False))

print("\nLand cover coverage summary:")
print(coverage_summary.to_string(index=False))

print("\nReview reason counts:")
print(
    historical_audit["review_reason"]
    .value_counts()
    .to_string()
)

print("\nSaved CSV files:")
for path in [
    all_records_path,
    review_path,
    year_path,
    coverage_path
]:
    print(path)

print("\nIMPORTANT NOTES:")
print("1. JMG dates remain unverified candidates.")
print("2. NASA COOLR dates also require source verification.")
print("3. Coordinate usability is a preliminary screening.")
print("4. Year overlap does not guarantee land cover pixel coverage.")
print("5. No records have been deleted or altered in raw files.")


Output directory: C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\inventory_audit_outputs
JMG records: 52
NASA COOLR records: 236

Combined record count: 288

HISTORICAL LAND COVER READINESS AUDIT
Total records: 288
JMG records: 52
NASA COOLR records: 236
Records with candidate dates: 283
Records with tentatively usable coordinates: 240
Records with land cover year overlap: 283

Year distribution:
source_dataset  event_year_candidate  record_count
           JMG                  2021            47
           JMG                  <NA>             5
    NASA_COOLR                  2007             6
    NASA_COOLR                  2008            15
    NASA_COOLR                  2009             9
    NASA_COOLR                  2010             9
    NASA_COOLR                  2011            22
    NASA_COOLR                  2012            14
    NASA_COOLR                  2013            28
    NASA_COOLR                  2014            33
    NASA_COOLR            

### 16. Candidate-year distribution and source coverage (NEW)

The year counts below reflect **candidate dates**, not certified occurrence dates. Use them to prioritise source verification and data downloads. Unknown dates are retained in a separate group.

In [18]:
# Export year distribution and period coverage summary.
year_distribution = (
    historical_audit.groupby(["source_dataset", "event_year_candidate"], dropna=False)
    .size().reset_index(name="record_count")
    .sort_values(["source_dataset", "event_year_candidate"], na_position="last")
)
year_file = AUDIT_DIR / "landslide_candidate_year_distribution.csv"
year_distribution.to_csv(year_file, index=False, encoding="utf-8-sig")

coverage_cols = [
    "figshare_1982_2015", "copernicus_2015_2019",
    "dynamic_world_2015_2025", "esri_2017_2025"
]
coverage_summary = historical_audit.groupby("source_dataset")[coverage_cols].sum().reset_index()
coverage_summary["total_records"] = historical_audit.groupby("source_dataset").size().values
coverage_summary["missing_or_invalid_date"] = (
    historical_audit.assign(missing=~historical_audit["date_available"])
    .groupby("source_dataset")["missing"].sum().values
)
coverage_file = AUDIT_DIR / "landslide_candidate_landcover_year_coverage_summary.csv"
coverage_summary.to_csv(coverage_file, index=False, encoding="utf-8-sig")

print("\nCandidate-year distribution (including missing dates):")
print(year_distribution.to_string(index=False))
print("\nYear-window coverage by source (windows can overlap):")
print(coverage_summary.to_string(index=False))
print("\nSaved:", year_file)
print("Saved:", coverage_file)
print("IMPORTANT: Historical rainfall matching needs independently confirmed event dates;")
print("land-cover year overlap does not imply spatial or cloud-free coverage.")


Candidate-year distribution (including missing dates):
source_dataset  event_year_candidate  record_count
           JMG                  2021            47
           JMG                  <NA>             5
    NASA_COOLR                  2007             6
    NASA_COOLR                  2008            15
    NASA_COOLR                  2009             9
    NASA_COOLR                  2010             9
    NASA_COOLR                  2011            22
    NASA_COOLR                  2012            14
    NASA_COOLR                  2013            28
    NASA_COOLR                  2014            33
    NASA_COOLR                  2015            29
    NASA_COOLR                  2016            23
    NASA_COOLR                  2017            35
    NASA_COOLR                  2018            12
    NASA_COOLR                  2022             1

Year-window coverage by source (windows can overlap):
source_dataset  figshare_1982_2015  copernicus_2015_2019  dynamic_world_2